In [ ]:
import pandas as pd
import numpy as np
import json
import os
import calendar
from datetime import datetime, date

# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V6  —  VT-Only Indent-Based Planning
#
#  Planning logic:
#    daily_indent  = monthly_indent / working_days
#    today_target  = max(0, daily_indent − inventory)
#    working_days  = calendar days in month − number of Sundays
#
#  Skip rules (part excluded from planning entirely):
#    • monthly_indent < 100
#    • monthly_indent / rate  ≤ 2 hours  (whole indent is trivial)
#
#  Zero-inventory rule:
#    If inventory = 0, part is FORCED into plan regardless of
#    skip rules. Parts sorted by highest daily_indent first
#    (greedy fill) so most critical parts always get time.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS  (change these every morning)
# -------------------------------------------------------------
# Only two lines to update each day:
#   PLANNING_DATE  : today
#   INDENT_MONTH   : only change when the month rolls over
# =============================================================

PLANNING_DATE = date(2026, 3, 14)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS    = 22        # machine hours available per shift
MIN_RUN_HOURS      = 4         # minimum run block per part
TARGET_DAYS_INV    = 3         # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Skip thresholds
MIN_MONTHLY_INDENT = 150       # skip if monthly indent < this
MIN_INDENT_HOURS   = 4.0       # skip if whole monthly indent takes ≤ this many hours
                               # (monthly_indent / rate ≤ MIN_INDENT_HOURS)
                               # EXCEPTION: zero-inventory parts are never skipped

# Priority: parts are sorted purely by daily_indent descending.
# Highest daily indent = most critical = planned first.
# Within same daily_indent, zero-inventory parts rank above others.

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS CALCULATION
# -------------------------------------------------------------
# working_days = calendar days in INDENT_MONTH − Sundays
# Example: March 2026 → 31 days, 5 Sundays → 26 working days
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*62}")
print(f"  Smart APS V6  —  VT-Only Indent-Based Planning")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Total days     : {TOTAL_DAYS}")
print(f"  Sundays        : {SUNDAY_COUNT}")
print(f"  Working days   : {WORKING_DAYS}  (used as divisor)")
print(f"  Daily target   : monthly_indent / {WORKING_DAYS}")
print(f"  Today target   : max(0, daily_indent - inventory)")
print(f"  Skip rule 1    : monthly_indent < {MIN_MONTHLY_INDENT} qty")
print(f"  Skip rule 2    : monthly_indent / rate  ≤ {MIN_INDENT_HOURS}h (trivial run)")
print(f"  Zero-inv rule  : inventory=0 parts always forced into plan")
print(f"{'='*62}\n")

# =============================================================
# SECTION 5 — LOAD DATA  (VT only)
# =============================================================

print("Loading data...")
stats        = pd.read_excel(book_path,       sheet_name="Sheet2")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# -------------------------------------------------------------
# Merge VT parts sheet with stats (cycle time, cavity, inventory).
# Rate (units/hour) = (3600 / cycle_time_seconds) × cavity_count
# =============================================================

def find_col(df, name, sheet):
    """Case-insensitive column finder. Raises clear error if missing."""
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'. "
            f"Available columns: {list(df.columns)}"
        )
    return match

stats_part_col = find_col(stats, "Part", "Sheet2")
vt_part_col    = find_col(vt_parts_raw, "Part", "VT")

data = stats.merge(
    vt_parts_raw.rename(columns={vt_part_col: "Material"}),
    left_on=stats_part_col,
    right_on="Material",
    how="inner"
).drop_duplicates(subset="Material").copy()

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()

print(f"  VT parts after merge with stats: {len(data)}")

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }

inventory = safe_dict(data, "Material", "Inventory on 24th")
rate      = safe_dict(data, "Material", "Rate")

# =============================================================
# SECTION 7A — READ INDENT FROM VT SHEET
# -------------------------------------------------------------
# VT sheet must have:
#   • "Part"   column  (case-insensitive)
#   • "Indent" column  (case-insensitive)
#
# Derived dicts:
#   indent_monthly   : {part → monthly qty}
#   indent_daily     : {part → monthly / working_days}
#   today_target_qty : {part → max(0, daily_indent − inventory)}
#
# Skip flags (computed here, enforced in scheduler):
#   skip_low_indent  : monthly < MIN_MONTHLY_INDENT
#   skip_trivial_run : monthly / rate ≤ MIN_INDENT_HOURS
#   EXCEPTION        : both skips are ignored if inventory == 0
# =============================================================

def read_indent_from_sheet(df, sheet_name):
    result   = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"),   None)
    ind_col  = next((c for c in df.columns if str(c).strip().lower() == "indent"), None)
    if part_col is None:
        print(f"  WARNING: 'Part' column not found in '{sheet_name}'. "
              f"Available: {list(df.columns)}")
        return result
    if ind_col is None:
        print(f"  WARNING: 'Indent' column not found in '{sheet_name}'. "
              f"Available: {list(df.columns)}")
        return result
    print(f"  Book1 '{sheet_name}' — Part col: '{part_col}'  Indent col: '{ind_col}'")
    for _, row in df.iterrows():
        part = row[part_col]
        val  = row[ind_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        result[str(part).strip()] = float(val) if pd.notna(val) else 0.0
    return result

print("\nReading indent data from VT sheet...")
indent_monthly = read_indent_from_sheet(vt_parts_raw, "VT")

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# ── Skip flags ───────────────────────────────────────────────
# A part is skipped ONLY when BOTH conditions below are true:
#   condition met  AND  inventory > 0
# If inventory == 0, part is NEVER skipped — it must be made.

def should_skip(part):
    """
    Returns (skip: bool, reason: str).
    Zero-inventory parts are never skipped.
    """
    inv     = inventory.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    # Zero inventory → always force into plan
    if inv == 0.0 and monthly > 0:
        return False, ""

    if monthly < MIN_MONTHLY_INDENT:
        return True, (f"Monthly indent {monthly:.0f} < {MIN_MONTHLY_INDENT} "
                      f"(threshold)")

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, (f"Whole monthly indent takes only {indent_hrs:.2f}h "
                      f"≤ {MIN_INDENT_HOURS}h threshold")

    return False, ""

# Print skip summary
skipped_parts = {p: should_skip(p) for p in indent_monthly}
n_skip  = sum(1 for skip, _ in skipped_parts.values() if skip)
n_force = sum(1 for p in indent_monthly
              if inventory.get(p, 0) == 0 and indent_monthly.get(p, 0) > 0)
print(f"\n  Indent summary:")
print(f"    Total VT parts with indent data : {len(indent_monthly)}")
print(f"    Parts skipped (low/trivial)     : {n_skip}")
print(f"    Parts with zero inventory       : {n_force}  (forced into plan regardless)")

for p, (skip, reason) in skipped_parts.items():
    if skip:
        print(f"    SKIP  {p:30s}  {reason}")

# =============================================================
# SECTION 7B — CHANGEOVER TIMES  (VT only)
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

print(f"\n  VT changeover times loaded: {len(vt_changeover)} machines")
for m, h in vt_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")

# =============================================================
# SECTION 7C — PART CATEGORY  (Runner / Repeater / Stranger)
# =============================================================

def build_category_from_sheet(df, sheet_name):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"),     None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        print(f"  WARNING: 'Part' column not found in '{sheet_name}'")
        return cat
    if cat_col is None:
        print(f"  WARNING: 'Category' column not found in '{sheet_name}'")
        return cat
    print(f"  Book1 '{sheet_name}' — Part col: '{part_col}'  Category col: '{cat_col}'")
    for _, row in df.iterrows():
        part = row[part_col]
        val  = str(row[cat_col]).strip() if pd.notna(row[cat_col]) else "Stranger"
        if pd.isna(part) or str(part).strip() == "":
            continue
        if val.lower() in ("runner", "repeater", "stranger"):
            val = val.capitalize()
        cat[str(part).strip()] = val
    return cat

part_category  = build_category_from_sheet(vt_parts_raw, "VT")
runner_count   = sum(1 for v in part_category.values() if v == "Runner")
repeater_count = sum(1 for v in part_category.values() if v == "Repeater")
stranger_count = sum(1 for v in part_category.values() if v == "Stranger")
print(f"  Part categories — Runner: {runner_count}  "
      f"Repeater: {repeater_count}  Stranger: {stranger_count}")

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"\n  Machine state loaded:")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    print("\n  Machine state: FIRST RUN — no previous state. "
          f"State will be saved to '{MACHINE_STATE_FILE}' after this run.")
    return {}

def save_machine_state(vt_state):
    combined = {m: p for m, p in vt_state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX  (VT only)
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — INDENT HORIZON TABLE
# -------------------------------------------------------------
# Shows per-part planning numbers and skip/force status.
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":              p,
            "Monthly_Indent":    round(monthly, 0),
            "Indent_Hrs_Total":  round(indent_hrs, 2),
            "Working_Days":      WORKING_DAYS,
            "Daily_Indent":      round(daily, 2),
            "Inventory_Now":     round(inv, 0),
            "Today_Target_Qty":  round(target, 0),
            "Today_Target_Hrs":  round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hr":       round(r, 2),
            "Indent_Status":     status,
            "Skip_Reason":       skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip:
            continue
        if daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — All active parts have zero indent today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, "SCENARIO 0 — ALL parts critical (inv < 1 day indent)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical (inv < 1 day indent)"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days inv)"

# =============================================================
# SECTION 12 — PRIORITY ORDERING
# -------------------------------------------------------------
# Parts are ranked purely by daily_indent descending.
# Highest daily indent = largest requirement = planned first.
# This ensures big parts always get machine time before small ones.
#
# Tie-break: zero-inventory parts rank above parts with some stock
# at the same daily_indent level (inv=0 → sort key slightly higher).
#
# "Inventory sufficient" parts (inv >= daily_indent) are excluded
# from the active list before this function is called.
# =============================================================

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []
    for p in parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        days_cov = inv / daily if daily > 0 else 999
        target   = horizon.loc[p, "Today_Target_Qty"] if p in horizon.index else 0

        # Sort key: daily_indent descending.
        # Zero-inventory parts get a tiny tie-break boost (+0.0001)
        # so they edge out same-daily_indent parts that have some stock.
        sort_key = daily + (0.0001 if inv == 0 else 0.0)

        rows.append({
            "Part":           p,
            "Days_Coverage":  round(days_cov, 2),
            "Inv_Now":        round(inv, 0),
            "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":   round(daily, 2),
            "Today_Target":   round(target, 0),
            "Sort_Key":       sort_key,
            "Zero_Inv":       "YES" if inv == 0 else "No",
        })

    return (pd.DataFrame(rows)
              .sort_values("Sort_Key", ascending=False)
              .drop(columns=["Sort_Key"])
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours,
                  machine_last_part, changeover_dict, inv_days):
    category    = part_category.get(part, "Stranger")
    is_runner   = (category == "Runner")
    runner_lock = is_runner and inv_days <= 1.0

    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue
        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue
        last   = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if last == part else changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = free - co_hrs
        if effective_free < MIN_RUN_HOURS:
            continue
        cost = co_hrs / AVAILABLE_HOURS + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, effective_free, co_hrs))

    ranked.sort(key=lambda x: x[1])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — TOOL-CHANGER STAGGER
# =============================================================

def stagger_changeovers(plan, machines, changeover_dict):
    print(f"\n  Tool-changer stagger:")

    def get_co_events(plan, machines):
        events = []
        for m in machines:
            m_rows = [r for r in plan if r["Machine"] == m]
            if len(m_rows) < 2:
                continue
            cursor = 0.0
            for i, row in enumerate(m_rows):
                co_h  = float(row.get("Changeover_Hrs") or 0.0)
                run_h = float(row.get("Run_Hours")       or 0.0)
                if co_h > 0:
                    events.append({
                        "machine":     m,
                        "part_before": m_rows[i-1]["Part"] if i > 0 else "—",
                        "part_after":  row["Part"],
                        "co_start":    cursor,
                        "co_duration": co_h,
                        "co_end":      cursor + co_h,
                        "row_before":  m_rows[i-1],
                        "row_after":   row,
                    })
                cursor += co_h + run_h
        events.sort(key=lambda e: e["co_start"])
        return events

    for _ in range(20):
        events = get_co_events(plan, machines)
        if not events:
            print("    No changeovers in plan — nothing to stagger")
            return

        conflict_found = False
        for i in range(1, len(events)):
            prev = events[i - 1]
            curr = events[i]
            if curr["co_start"] >= prev["co_end"]:
                continue

            conflict_found = True
            overlap = prev["co_end"] - curr["co_start"]

            row_bc   = curr["row_before"]
            run_h_bc = float(row_bc.get("Run_Hours") or 0.0)
            feasible_A = (run_h_bc - overlap) >= MIN_RUN_HOURS

            row_bp   = prev["row_before"]
            m_prev   = prev["machine"]
            used_p   = sum(
                float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
                for r in plan if r["Machine"] == m_prev
            )
            spare_p  = AVAILABLE_HOURS - used_p
            shift_B  = min(overlap, spare_p)
            feasible_B = shift_B > 0

            if feasible_A and feasible_B:
                use_A = (overlap * rate.get(row_bc["Part"], 1)) <= (0 if spare_p >= overlap else
                          (overlap - spare_p) * rate.get(row_bp["Part"], 1))
            elif feasible_A:
                use_A = True
            elif feasible_B:
                use_A = False
            else:
                print("    ⚠ Cannot resolve conflict — insufficient capacity to shift")
                continue

            if use_A:
                new_h = round(run_h_bc - overlap, 3)
                row_bc["Run_Hours"]      = new_h
                row_bc["Production_Qty"] = round(
                    row_bc["Production_Qty"] - overlap * rate.get(row_bc["Part"], 1), 0)
                row_bc["Total_Hrs_Used"] = round(
                    float(row_bc.get("Changeover_Hrs") or 0) + new_h, 3)
                row_bc["Stagger_Adjusted"] = f"Reduced {round(overlap*60,1)}min (Option A)"
                print(f"    ✓ {curr['machine']:15s} conflict resolved (Option A)")
            else:
                new_h = round(float(row_bp.get("Run_Hours") or 0) + shift_B, 3)
                row_bp["Run_Hours"]      = new_h
                row_bp["Production_Qty"] = round(
                    row_bp["Production_Qty"] + shift_B * rate.get(row_bp["Part"], 1), 0)
                row_bp["Total_Hrs_Used"] = round(
                    float(row_bp.get("Changeover_Hrs") or 0) + new_h, 3)
                row_bp["Stagger_Adjusted"] = f"Extended {round(shift_B*60,1)}min (Option B)"
                print(f"    ✓ {prev['machine']:15s} conflict resolved (Option B)")
            break

        if not conflict_found:
            print("    All changeovers staggered — no conflicts  ✓")
            return

    print(f"  ⚠ Stagger did not fully resolve in 20 passes — check manually")

# =============================================================
# SECTION 14B — 22-HOUR FILLER
# =============================================================

def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df):
    already_planned = {row["Part"] for row in plan}

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining <= 0:
            continue

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # Option A — extend existing part: pick the one with highest daily_indent
        # (most critical part on this machine gets the extra time)
        best_part, best_daily = None, -1
        for p in on_machine:
            d = indent_daily.get(p, 0)
            if d > best_daily:
                best_daily, best_part = d, p

        if best_part and best_daily > 0:
            p         = best_part
            r_val     = rate.get(p, 1)
            extra_qty = round(remaining * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(row["Run_Hours"] + remaining, 2)
                    row["Production_Qty"] = round(row["Production_Qty"] + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        row.get("Total_Hrs_Used", row["Run_Hours"]) + remaining, 2)
                    row["Type"] = "Primary+Extended"
                    break
            machine_hours[m]    += remaining
            current_inventory[p] = current_inventory.get(p, 0) + extra_qty
            print(f"    ↑ {p:30s} → {m:15s}  +{remaining:.2f}h merged  "
                  f"qty+={extra_qty:.0f}  [EXTEND]")
            continue

        # Option B — add a new part (≥ MIN_RUN_HOURS required)
        if remaining < MIN_RUN_HOURS:
            continue

        candidates = []
        for p in parts:
            if p in already_planned:
                continue
            if m not in compatibility.get(p, []):
                continue
            daily   = indent_daily.get(p, 0)
            inv_now  = current_inventory.get(p, 0)
            # Skip if inventory already sufficient
            if inv_now >= daily > 0:
                continue
            candidates.append((p, daily))

        # Sort by daily_indent descending — biggest parts fill first
        candidates.sort(key=lambda x: -x[1])

        for p, daily_val in candidates:
            last    = machine_last_part.get(m)
            co_hrs  = 0.0 if last == p else vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            eff_run = remaining - co_hrs
            if eff_run < MIN_RUN_HOURS:
                continue
            r_val = rate.get(p, 1)
            qty   = round(eff_run * r_val, 0)
            machine_hours[m]    += (co_hrs + eff_run)
            current_inventory[p] = current_inventory.get(p, 0) + qty
            machine_last_part[m] = p
            already_planned.add(p)
            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(eff_run, 2),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + eff_run, 2),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(indent_daily.get(p, 0), 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "New — filler",
                "Runner_Lock":      "No",
                "Priority_Score":   0,
                "Stagger_Adjusted": "No",
            })
            print(f"    + {p:30s} → {m:15s}  {eff_run:.2f}h  qty={qty:.0f}  "
                  f"[Filler]  CO={'Yes' if co_hrs > 0 else 'No'}")
            break

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, changeover_dict, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    scenario, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # Print what needs production
    prod_needed = horizon_df[
        horizon_df["Indent_Status"].isin(["PRODUCTION NEEDED", "ZERO INV — FORCED"])
    ]
    if not prod_needed.empty:
        print(f"\n  Parts requiring production today ({len(prod_needed)}):")
        print(f"  {'Part':<30}  {'Monthly':>8}  {'Daily':>7}  {'Inv':>8}  "
              f"{'Target':>8}  {'Status'}")
        print(f"  {'─'*30}  {'─'*8}  {'─'*7}  {'─'*8}  {'─'*8}  {'─'*20}")
        for _, r in prod_needed.iterrows():
            print(f"  {r['Part']:<30}  "
                  f"{r['Monthly_Indent']:>8.0f}  "
                  f"{r['Daily_Indent']:>7.2f}  "
                  f"{r['Inventory_Now']:>8.0f}  "
                  f"{r['Today_Target_Qty']:>8.0f}  "
                  f"{r['Indent_Status']}")
    else:
        print("  No production needed — inventory covers all active parts today")

    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned, skipped_log = [], [], [], []

    # Active parts = not skipped OR zero-inventory (always forced)
    # Inventory-sufficient parts are excluded here — not in priority list at all.
    active_parts = [
        p for p in parts
        if (not should_skip(p)[0] or inventory.get(p, 0) == 0)
        and indent_monthly.get(p, 0) > 0
        and not (inventory.get(p, 0) >= indent_daily.get(p, 0) > 0)
    ]

    priority_df = compute_priority(active_parts, horizon_df)

    print(f"\n  Priority order — sorted by daily_indent descending "
          f"({len(priority_df)} active parts):")
    print(f"  {'Part':<30}  {'Daily':>7}  {'Inv':>8}  {'Target':>8}  {'Zero_Inv':>8}")
    print(f"  {'─'*30}  {'─'*7}  {'─'*8}  {'─'*8}  {'─'*8}")
    for _, r in priority_df.iterrows():
        print(f"  {r['Part']:<30}  "
              f"{r['Daily_Indent']:>7.2f}  "
              f"{r['Inv_Now']:>8.0f}  "
              f"{r['Today_Target']:>8.0f}  "
              f"{r['Zero_Inv']:>8}")

    runner_dedicated_machines = set()

    # ── Log skipped parts (not zero-inv) ─────────────────────
    for p in parts:
        skip, reason = should_skip(p)
        inv_now = inventory.get(p, 0)
        if skip and inv_now > 0:
            skipped_log.append({
                "Part":           p,
                "Category":       part_category.get(p, "Stranger"),
                "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":   round(indent_daily.get(p, 0), 2),
                "Inventory_Now":  round(inv_now, 0),
                "Skip_Reason":    reason,
            })

    def run_assignment(parts_subset, pass_label, exclude_machines=None):
        if exclude_machines is None:
            exclude_machines = set()
        print(f"\n  {pass_label}:")

        for _, row in priority_df[priority_df["Part"].isin(parts_subset)].iterrows():
            part     = row["Part"]

            if any(r["Part"] == part for r in plan):
                print(f"    ~ {part:30s}  ALREADY PLANNED — skipped")
                continue

            inv_now  = current_inventory.get(part, 0)
            daily    = indent_daily.get(part, 0)
            monthly  = indent_monthly.get(part, 0)
            r_val    = rate.get(part, 1)
            category = part_category.get(part, "Stranger")
            inv_days = inv_now / daily if daily > 0 else 999
            target   = today_target_qty.get(part, 0)

            compatible_mch = compatibility.get(part, [])

            # ── Zero indent ───────────────────────────────────
            if monthly == 0:
                deferred.append({
                    "Part":           part,
                    "Category":       category,
                    "Inventory_Now":  round(inv_now, 0),
                    "Monthly_Indent": 0,
                    "Daily_Indent":   0,
                    "Today_Target":   0,
                    "Days_Coverage":  round(inv_days, 2),
                    "Reason":         "Monthly indent is zero",
                    "Next_Action":    "Re-evaluate when indent is set",
                })
                print(f"    - {part:30s} [{category:8s}]  DEFERRED — indent = 0")
                continue

            # ── Inventory sufficient (and not zero-inv forced) ─
            if target == 0 and inv_now > 0:
                deferred.append({
                    "Part":           part,
                    "Category":       category,
                    "Inventory_Now":  round(inv_now, 0),
                    "Monthly_Indent": round(monthly, 0),
                    "Daily_Indent":   round(daily, 2),
                    "Today_Target":   0,
                    "Days_Coverage":  round(inv_days, 2),
                    "Reason":         "Inventory covers daily indent — no production needed",
                    "Next_Action":    "Re-evaluate tomorrow",
                })
                print(f"    - {part:30s} [{category:8s}]  DEFERRED  "
                      f"inv={inv_now:.0f} ≥ daily={daily:.2f}")
                continue

            # ── Target hours ──────────────────────────────────
            # For zero-inventory forced parts: use daily_indent as minimum
            # run target so we at least build one day's worth.
            effective_target = max(target, daily) if inv_now == 0 else target

            if category == "Runner":
                target_hrs = min(
                    (effective_target / r_val if r_val > 0 else MIN_RUN_HOURS)
                    + (daily / r_val if r_val > 0 else 0),
                    AVAILABLE_HOURS
                )
                target_hrs = max(target_hrs, MIN_RUN_HOURS)
            else:
                target_hrs = max(
                    MIN_RUN_HOURS,
                    effective_target / r_val if r_val > 0 else MIN_RUN_HOURS
                )

            # ── No compatible machines ────────────────────────
            if not compatible_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": "NONE DEFINED",
                    "Reason":              "Part has no compatible machines in VT matrix",
                    "Action_Needed":       "Add part to VT_Matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — not in matrix")
                continue

            available_mch = [m for m in compatible_mch if m not in exclude_machines]
            if not available_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              "All compatible machines dedicated to Runners",
                    "Action_Needed":       "Add more compatible machines in matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — machines Runner-dedicated")
                continue

            ranked, runner_lock = rank_machines(
                part, available_mch, compatibility,
                machine_hours, machine_last_part,
                changeover_dict, inv_days
            )

            if not ranked:
                free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                            for m in available_mch if m in machine_hours}
                if runner_lock:
                    last_m = next(
                        (m for m in available_mch if machine_last_part.get(m) == part), None)
                    free_l = round(AVAILABLE_HOURS - machine_hours.get(last_m, 0), 2) if last_m else 0
                    reason = (f"RUNNER (inv≤1d) — must stay on {last_m}, "
                              f"only {free_l}h free") if last_m else \
                             "RUNNER — no machine state (first run)"
                    action = f"Free capacity on {last_m}" if last_m else \
                             "Machine state saves after this run"
                elif all(h < MIN_RUN_HOURS for h in free_map.values()):
                    reason = ("All compatible machines full. Free: "
                              + ", ".join(f"{m}={h}h" for m, h in free_map.items()))
                    action = "Reduce lower-priority part hours or plan tomorrow"
                else:
                    reason = ("Remaining hrs < MIN after changeover. Free: "
                              + ", ".join(f"{m}={h}h" for m, h in free_map.items()))
                    action = "Check Inventory_Health sheet"

                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       action,
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — {reason[:60]}")
                continue

            # ── Assign ────────────────────────────────────────
            assigned = False
            for m, cost, effective_free, co_hrs in ranked:
                run_h = min(target_hrs, effective_free)
                run_h = max(run_h, MIN_RUN_HOURS)
                run_h = min(run_h, effective_free)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += (co_hrs + run_h)
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part

                if category == "Runner":
                    runner_dedicated_machines.add(m)

                forced_tag = "ZERO-INV-FORCED" if inv_now == 0 else "Primary"
                plan.append({
                    "Part":              part,
                    "Category":          category,
                    "Machine":           m,
                    "Run_Hours":         round(run_h, 2),
                    "Changeover_Hrs":    round(co_hrs, 3),
                    "Total_Hrs_Used":    round(co_hrs + run_h, 2),
                    "Rate_Per_Hour":     round(r_val, 2),
                    "Production_Qty":    qty,
                    "Monthly_Indent":    round(monthly, 0),
                    "Daily_Indent":      round(daily, 2),
                    "Today_Target":      round(target, 0),
                    "Changeover":        "No" if co_hrs == 0 else "Yes",
                    "Type":              forced_tag,
                    "Runner_Lock":       "YES" if runner_lock else "No",
                    "Priority_Score":    round(indent_daily.get(part, 0), 2),
                    "Stagger_Adjusted":  "No",
                })
                co_str = "No" if co_hrs == 0 else f"Yes ({co_hrs*60:.0f}min)"
                tag    = "  [DEDICATED]" if category == "Runner" else \
                         "  [ZERO-INV FORCED]" if inv_now == 0 else ""
                print(f"    ✓ {part:30s} [{category:8s}] → {m:15s}  "
                      f"{run_h:.2f}h  qty={qty:>8.0f}  CO={co_str}{tag}")
                assigned = True
                break

            if not assigned:
                free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                            for m in available_mch if m in machine_hours}
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              (
                        f"No single machine has {round(target_hrs,2)}h free after CO. "
                        + "Free: " + ", ".join(f"{m}={h}h" for m, h in free_map.items())
                    ),
                    "Action_Needed": "Reduce lower-priority hours or verify inventory covers gap",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — "
                      f"no machine has {round(target_hrs,2)}h free")

    # PASS 1 — Runners
    runner_parts     = [p for p in parts
                        if part_category.get(p, "Stranger") == "Runner"
                        and (not should_skip(p)[0] or inventory.get(p, 0) == 0)]
    non_runner_parts = [p for p in parts
                        if part_category.get(p, "Stranger") != "Runner"
                        and (not should_skip(p)[0] or inventory.get(p, 0) == 0)]

    run_assignment(runner_parts,
                   "PASS 1 — Runners (dedicated machine, cap = target + 1-day buffer)")
    print(f"\n  Runner-dedicated machines: "
          f"{sorted(runner_dedicated_machines) if runner_dedicated_machines else 'none'}")

    # PASS 2 — Repeaters + Strangers
    run_assignment(non_runner_parts,
                   "PASS 2 — Repeaters + Strangers (Runner machines excluded)",
                   exclude_machines=runner_dedicated_machines)

    # 22-hour filler
    print(f"\n  22-hour filler:")
    fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df
    )

    # Duplicate check
    part_counts = {}
    for row in plan:
        part_counts[row["Part"]] = part_counts.get(row["Part"], 0) + 1
    dupes = {p: c for p, c in part_counts.items() if c > 1}
    if dupes:
        print(f"  WARNING: duplicate parts: {dupes}")
    else:
        print(f"    No duplicate parts — each part appears exactly once  ✓")

    # Stagger
    stagger_changeovers(plan, machines, changeover_dict)

    # ── Inventory health ──────────────────────────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Monthly_Indent":  round(monthly, 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"      if days_cov >= 1
                                else "CRITICAL"),
        })

    # ── Machine utilization ───────────────────────────────────
    mach_rows = []
    for m in machines:
        used      = machine_hours.get(m, 0)
        parts_run = [r["Part"] for r in plan if r["Machine"] == m]
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        util_status = ("FULL"     if used >= AVAILABLE_HOURS - 0.5 else
                       "GOOD"     if used >= AVAILABLE_HOURS * 0.85 else
                       "PARTIAL"  if used >= AVAILABLE_HOURS * 0.5 else
                       "UNDERUSED")
        mach_rows.append({
            "Machine":              m,
            "Total_Available_Hrs":  AVAILABLE_HOURS,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               util_status,
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    plan_df     = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    skip_df     = pd.DataFrame(skipped_log)  if skipped_log  else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df["Category"] = plan_df["Part"].map(
            lambda p: part_category.get(p, "Stranger"))
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    return plan_df, def_df, not_df, skip_df, mach_df, inv_df, machine_last_part, horizon_df

# =============================================================
# SECTION 16 — RUN VT SCHEDULER
# =============================================================

vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

(vt_plan, vt_def, vt_not, vt_skip,
 vt_mach, vt_inv, vt_state, vt_horizon) = schedule(
    vt_parts, vt_compat, vt_machines, vt_changeover, "VT Machines"
)

save_machine_state(vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":    "0D6E6E",
    "VT_Plan":               "1F4E79",
    "VT_Machine_Util":       "375623",
    "VT_Not_Planned":        "7B2C2C",
    "VT_Not_Required_Today": "7F6000",
    "VT_Skipped_Parts":      "5C3D2E",
    "VT_Inventory_Health":   "4A235A",
    "VT_Indent_Horizon":     "154360",
}

STATUS_FILLS = {
    "FULL":               PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":               PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":            PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":          PatternFill("solid", fgColor="FFC7CE"),
    "OK":                 PatternFill("solid", fgColor="C6EFCE"),
    "LOW":                PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":           PatternFill("solid", fgColor="FFC7CE"),
    "PRODUCTION NEEDED":  PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":  PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":     PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":            PatternFill("solid", fgColor="EDEDED"),
    "NO INDENT":          PatternFill("solid", fgColor="EDEDED"),
}

def style_sheet(ws, header_hex):
    hf = PatternFill("solid", fgColor=header_hex)
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and ("Status" in str(col_name) or "Indent_Status" in str(col_name)):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def build_machine_wise_plan(plan_df, machines):
    """
    Build the machine-wise sequenced plan with Rate_Per_Hour column included.
    One row per part per machine, plus a summary footer row per machine.
    """
    if plan_df.empty:
        return pd.DataFrame()

    def safe_float(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    def fmt_time(h):
        try:
            total_min = int(round(safe_float(h, 0.0) * 60))
            return f"{total_min // 60:02d}:{total_min % 60:02d}"
        except Exception:
            return "??"

    rows = []
    for m in machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        cumulative_hrs = 0.0
        seq            = 1

        for _, pr in machine_rows.iterrows():
            part    = pr.get("Part", "—")
            run_h   = safe_float(pr.get("Run_Hours", 0))
            co_h    = safe_float(pr.get("Changeover_Hrs", 0))
            r_val   = safe_float(pr.get("Rate_Per_Hour", 0))
            co_mins = round(co_h * 60, 1)
            start_h = cumulative_hrs + co_h
            end_h   = start_h + run_h

            rows.append({
                "Machine":                m,
                "Seq":                    seq,
                "Part":                   part,
                "Category":               part_category.get(part, "Stranger"),
                "Rate_Per_Hour":          round(r_val, 2),          # ← added
                "Changeover_Before_Mins": co_mins,
                "Run_Hours":              round(run_h, 2),
                "Start_Time":             fmt_time(cumulative_hrs),
                "Start_After_CO":         fmt_time(start_h),
                "End_Time":               fmt_time(end_h),
                "Cumulative_Hrs":         round(end_h, 2),
                "Production_Qty":         safe_float(pr.get("Production_Qty", 0)),
                "Monthly_Indent":         safe_float(pr.get("Monthly_Indent", 0)),
                "Daily_Indent":           safe_float(pr.get("Daily_Indent", 0)),
                "Today_Target":           safe_float(pr.get("Today_Target", 0)),
                "Changeover":             pr.get("Changeover", "No") or "No",
                "Type":                   pr.get("Type", "Primary") or "Primary",
                "Runner_Lock":            pr.get("Runner_Lock", "No") or "No",
                "Row_Type":               "Part",
            })
            cumulative_hrs = end_h
            seq           += 1

        # Summary footer
        total_used     = round(cumulative_hrs, 2)
        co_hrs_series  = machine_rows["Changeover_Hrs"].apply(lambda x: safe_float(x, 0))
        total_co_mins  = round(co_hrs_series.sum() * 60, 1)
        total_prod_hrs = round(total_used - co_hrs_series.sum(), 2)
        total_qty      = machine_rows["Production_Qty"].apply(lambda x: safe_float(x, 0)).sum()
        co_count       = int(machine_rows["Changeover"].eq("Yes").sum())

        rows.append({
            "Machine":                m,
            "Seq":                    "—",
            "Part":                   f"TOTAL — {m}",
            "Category":               "—",
            "Rate_Per_Hour":          "—",
            "Changeover_Before_Mins": total_co_mins,
            "Run_Hours":              total_prod_hrs,
            "Start_Time":             "00:00",
            "Start_After_CO":         "—",
            "End_Time":               fmt_time(total_used),
            "Cumulative_Hrs":         total_used,
            "Production_Qty":         round(total_qty, 0),
            "Monthly_Indent":         "—",
            "Daily_Indent":           "—",
            "Today_Target":           "—",
            "Changeover":             f"{co_count} changeovers",
            "Type":                   f"Used {total_used}h / {AVAILABLE_HOURS}h  |  Unused {round(AVAILABLE_HOURS-total_used,2)}h",
            "Runner_Lock":            "—",
            "Row_Type":               "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_font  = Font(bold=True, color="FFFFFF", size=11)
    summary_fill = PatternFill("solid", fgColor="0D9488")
    summary_font = Font(bold=True, color="FFFFFF", size=11)
    part_fills   = [
        PatternFill("solid", fgColor="EFF6FF"),
        PatternFill("solid", fgColor="F0FDF4"),
    ]
    co_fill = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None

    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = summary_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            bg = part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = bg
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col - 1].value == "Yes":
                row[co_col - 1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(40, max_len + 3))
    ws.freeze_panes = "C2"


# =============================================================
# SECTION 17A — WRITE EXCEL
# =============================================================

print(f"\nWriting → {output_path}")

vt_mw = build_machine_wise_plan(vt_plan, vt_machines)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    vt_mw.to_excel(writer,      sheet_name="VT_Plan_By_Machine",    index=False)
    vt_plan.to_excel(writer,    sheet_name="VT_Plan",               index=False)
    vt_mach.to_excel(writer,    sheet_name="VT_Machine_Util",       index=False)
    vt_not.to_excel(writer,     sheet_name="VT_Not_Planned",        index=False)
    vt_def.to_excel(writer,     sheet_name="VT_Not_Required_Today", index=False)
    vt_skip.to_excel(writer,    sheet_name="VT_Skipped_Parts",      index=False)
    vt_inv.to_excel(writer,     sheet_name="VT_Inventory_Health",   index=False)
    vt_horizon.to_excel(writer, sheet_name="VT_Indent_Horizon",     index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name != "VT_Plan_By_Machine":
        style_sheet(wb[sheet_name], header_hex)

tab_colors = {
    "VT_Plan_By_Machine":    "0D6E6E",
    "VT_Plan":               "1F4E79",
    "VT_Machine_Util":       "375623",
    "VT_Not_Planned":        "7B2C2C",
    "VT_Not_Required_Today": "7F6000",
    "VT_Skipped_Parts":      "5C3D2E",
    "VT_Inventory_Health":   "4A235A",
    "VT_Indent_Horizon":     "154360",
}
for name, color in tab_colors.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V6 Complete  —  VT Only  —  {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*62}")
print(f"  planned            = {len(vt_plan):>4}")
print(f"  not_required_today = {len(vt_def):>4}  (inv covers daily indent)")
print(f"  skipped            = {len(vt_skip):>4}  (indent < {MIN_MONTHLY_INDENT} qty  OR  whole indent ≤ {MIN_INDENT_HOURS}h)")
print(f"  not_planned        = {len(vt_not):>4}  (needed but no machine capacity)")
print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Update SECTION 1 each morning:")
print(f"        PLANNING_DATE = date(2026, 3, 15)")
print(f"        INDENT_MONTH  = date(2026, 3,  1)   ← only change when month rolls over")